# DFT XC skeleton 二阶导数分解 (TPSS0, MGGA)


In [1]:
from pyscf import gto, dft, lib
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")


In [2]:
import sys
sys.path.append("..")

from pyhessref.nimatmul.becke_partition import becke_partition
from pyhessref.nimatmul import rks as rks_nimatmul


In [3]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()


In [4]:
mf = dft.RKS(mol, xc="TPSS0").density_fit()
dat0 = np.load("nh3_r_tpss0.npz")
mf.mo_coeff = dat0["mo_coeff"]
mf.mo_occ = dat0["mo_occ"]
mf.mo_energy = dat0["mo_energy"]
mf.with_df.build()
mf.converged = True


In [5]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mocc = mo_coeff[:, mo_occ > 0]
dm0 = mocc @ mocc.T * 2
natm = mol.natm
nao = mol.nao
aoslices = mol.aoslice_by_atom()
ni = dft.numint.NumInt()


In [6]:
grids = dft.gen_grid.Grids(mol)
grids.build(sort_grids=False)
coords = grids.coords
weights = grids.weights
ngrids = len(weights)


In [7]:
# Reference de_vxc from 06-1: this is what we want to reproduce.
de_ks_ref = np.load("nh3_r_tpss0_decomp.npz")["de_vxc"]
print("de_vxc_ref shape:", de_ks_ref.shape)
print("de_vxc_ref fp:   ", lib.fp(de_ks_ref))


de_vxc_ref shape: (4, 4, 3, 3)
de_vxc_ref fp:    -0.8310821568119837


In [8]:
ao = ni.eval_ao(mol, grids.coords, deriv=3)
rho = ni.eval_rho2(mol, ao, mo_coeff, mo_occ, xctype="MGGA")
rho = rho[[0, 1, 2, 3, 5]]
print("rho shape:", rho.shape)


rho shape: (5, 43328)


In [9]:
xc_eff = ni.eval_xc_eff(mf.xc, rho, deriv=2, xctype="MGGA")
vxc = xc_eff[1]  # shape [5, ngrid]
fxc = xc_eff[2]  # shape [5, 5, ngrid]
print("vxc shape:", vxc.shape, "fxc shape:", fxc.shape)


vxc shape: (5, 43328) fxc shape: (5, 5, 43328)


In [10]:
TX, TY, TZ = 0, 1, 2
O = 0
X, Y, Z = 1, 2, 3
XX, XY, XZ = 4, 5, 6
YX, YY, YZ = 5, 7, 8
ZX, ZY, ZZ = 6, 8, 9
XXX, XXY, XXZ, XYY, XYZ, XZZ = 10, 11, 12, 13, 14, 15
YYY, YYZ, YZZ, ZZZ = 16, 17, 18, 19


In [11]:
ao_dm0 = ao @ dm0
ao_dm0.shape


(20, 43328, 49)

### fxc contribution


In [12]:
# TPSS0 (MGGA): tau channel present, drho has 5 components (RHO, GRAD_X, GRAD_Y, GRAD_Z, TAU)
drho = np.zeros((natm, 3, 5, ngrids))
for A in range(natm):
    _, _, p0, p1 = aoslices[A]
    slc = slice(p0, p1)
    ao_slc = ao[:, :, slc]
    ao_dm0_slc = ao_dm0[:, :, slc]
    DERIV_COMPONENTS = [
        # RHO part
        [(TX, 0), (X, O)],
        [(TY, 0), (Y, O)],
        [(TZ, 0), (Z, O)],
        # SIGMA part (bra deriv 2)
        [(TX, X), (XX, O)],
        [(TX, Y), (XY, O)],
        [(TX, Z), (XZ, O)],
        [(TY, X), (YX, O)],
        [(TY, Y), (YY, O)],
        [(TY, Z), (YZ, O)],
        [(TZ, X), (ZX, O)],
        [(TZ, Y), (ZY, O)],
        [(TZ, Z), (ZZ, O)],
        # SIGMA part (bra deriv 1, ket deriv 1)
        [(TX, X), (X, X)],
        [(TX, Y), (X, Y)],
        [(TX, Z), (X, Z)],
        [(TY, X), (Y, X)],
        [(TY, Y), (Y, Y)],
        [(TY, Z), (Y, Z)],
        [(TZ, X), (Z, X)],
        [(TZ, Y), (Z, Y)],
        [(TZ, Z), (Z, Z)],
        # TAU part (bra deriv 2, ket deriv 1)
        [(TX, 4), (XX, X)],
        [(TX, 4), (XY, Y)],
        [(TX, 4), (XZ, Z)],
        [(TY, 4), (YX, X)],
        [(TY, 4), (YY, Y)],
        [(TY, 4), (YZ, Z)],
        [(TZ, 4), (ZX, X)],
        [(TZ, 4), (ZY, Y)],
        [(TZ, 4), (ZZ, Z)],
    ]
    for ((t, v), (cbra, cket)) in DERIV_COMPONENTS:
        drho[A, t, v] -= np.einsum("gu, gu -> g", ao_slc[cbra], ao_dm0_slc[cket])
# scale symmetric coeff: RHO and SIGMA (0..3) get *2, TAU (4) does not
drho[:, :, :4] *= 2


In [13]:
lib.fp(drho)


np.float64(13922827.878215577)

In [14]:
de_fxc = np.einsum("g, Atxg, xyg, Bsyg -> ABts", weights, drho, fxc, drho)
print(lib.fp(de_fxc))


-29.390069496788005


### dao_vxc_diag contribution


In [15]:
# --- dao_vxc_diag (MGGA: with tau) --- #
dao_vxc_diag = np.zeros((6, nao))  # 6 denotes xx, xy, xz, yy, yz, zz
wv = weights * vxc  # [5, ngrids]

# Contribution 1: ao[i+4]^T @ (wv[0]*ao[0] + wv[1]*ao[1] + wv[2]*ao[2] + wv[3]*ao[3])
aow_diag = (np.einsum("gu, g -> gu", ao_dm0[0], wv[0])
          + np.einsum("gu, g -> gu", ao_dm0[1], wv[1])
          + np.einsum("gu, g -> gu", ao_dm0[2], wv[2])
          + np.einsum("gu, g -> gu", ao_dm0[3], wv[3]))
for idx, its in enumerate([XX, XY, XZ, YY, YZ, ZZ]):
    dao_vxc_diag[idx] += 2 * np.einsum("gu, gu -> u", ao[its], aow_diag)

# Contribution 2 (GGA triple-derivative part)
TRIPLE_DERIV_DIAG = [
    [XXX, XXY, XXZ],  # xx
    [XXY, XYY, XYZ],  # xy
    [XXZ, XYZ, XZZ],  # xz
    [XYY, YYY, YYZ],  # yy
    [XYZ, YYZ, YZZ],  # yz
    [XZZ, YZZ, ZZZ],  # zz
]
for idx, (i3x, i3y, i3z) in enumerate(TRIPLE_DERIV_DIAG):
    aow_triple = (np.einsum("gu, g -> gu", ao[i3x], wv[1])
                + np.einsum("gu, g -> gu", ao[i3y], wv[2])
                + np.einsum("gu, g -> gu", ao[i3z], wv[3]))
    dao_vxc_diag[idx] += 2 * np.einsum("gu, gu -> u", aow_triple, ao_dm0[0])

# Contribution 3 (TAU triple-derivative part)
aow_diag_tau = [np.einsum("gu, g -> gu", ao_dm0[d], wv[4]) for d in [X, Y, Z]]
TAU_DIAG_COMPONENTS = [
    ([XXX, XXY, XXZ, XYY, XYZ, XZZ], 0),  # direction x: ao[triple]^T @ aow_tau_x
    ([XXY, XYY, XYZ, YYY, YYZ, YZZ], 1),  # direction y
    ([XXZ, XYZ, XZZ, YYZ, YZZ, ZZZ], 2),  # direction z
]
for i, (triple_indices, direction) in enumerate(TAU_DIAG_COMPONENTS):
    for idx, j in enumerate(triple_indices):
        dao_vxc_diag[idx] += np.einsum("gu, gu -> u", ao[j], aow_diag_tau[direction])

de_vxc_diag = np.zeros((natm, natm, 6))
for A in range(natm):
    _, _, p0A, p1A = aoslices[A]
    slcA = slice(p0A, p1A)
    de_vxc_diag[A, A] += np.einsum("Au -> A", dao_vxc_diag[:, slcA])
de_vxc_diag = de_vxc_diag[:, :, [[0, 1, 2], [1, 3, 4], [2, 4, 5]]]
print("de_vxc_diag fp:", lib.fp(de_vxc_diag))


de_vxc_diag fp: 44.68386358957358


### dao_vxc_off contribution


In [16]:
# --- dao_vxc (MGGA: with tau) --- #
wv = weights * vxc  # [5, ngrids]
dao_vxc = np.zeros((3, 3, nao, nao))

# GGA part (RHO + SIGMA)
GGA_CALLS = [[XX, XY, XZ], [YX, YY, YZ], [ZX, ZY, ZZ]]

aowv = [None, None, None]
for t in range(3):
    aowv[t] = 0.5 * np.einsum("gu, g -> gu", ao[t + 1], wv[0])
    for r in range(3):
        aowv[t] += np.einsum("gu, g -> gu", ao[GGA_CALLS[t][r]], wv[r + 1])

for t in range(3):
    for s in range(3):
        dao_vxc[t, s] += 2 * aowv[s].T @ ao[t + 1]     # ipip[t,s]

# TAU part: ipip from three _d1d2_dot_ calls with wv[4]
aowv_tau = [np.einsum("gu, g -> gu", ao[4 + i], wv[4]) for i in range(6)]

TAU_CALLS = [
    ([0, 1, 2], [XX, XY, XZ]),  # {aow_tau[XX], aow_tau[XY], aow_tau[XZ]} vs {ao[XX], ao[XY], ao[XZ]}
    ([1, 3, 4], [YX, YY, YZ]),  # {aow_tau[XY], aow_tau[YY], aow_tau[YZ]} vs {ao[YX], ao[YY], ao[YZ]}
    ([2, 4, 5], [ZX, ZY, ZZ]),  # {aow_tau[XZ], aow_tau[YZ], aow_tau[ZZ]} vs {ao[ZX], ao[ZY], ao[ZZ]}
]

dao_vxc_tau = np.zeros((3, 3, nao, nao))

for r_bra, r_ket in TAU_CALLS:
    for t in range(3):
        for s in range(t + 1):
            dao_vxc_tau[t, s] += 0.5 * aowv_tau[r_bra[s]].T @ ao[r_ket[t]]    # ipip[d1,d2]

for t in range(3):
    for s in range(t):
        dao_vxc_tau[s, t] = dao_vxc_tau[t, s].T

dao_vxc += dao_vxc_tau
dao_vxc += dao_vxc.transpose(1, 0, 3, 2)  # [s,t] with AO indices transposed

de_vxc_off = np.zeros((natm, natm, 3, 3))
for A in range(natm):
    _, _, p0A, p1A = aoslices[A]
    slcA = slice(p0A, p1A)
    for B in range(A + 1):
        _, _, p0B, p1B = aoslices[B]
        slcB = slice(p0B, p1B)
        de_vxc_off[A, B] += np.einsum("tsuv, uv -> ts", dao_vxc[:, :, slcB, slcA], dm0[slcB, slcA])
        if A != B:
            de_vxc_off[B, A] = de_vxc_off[A, B].T
print("de_vxc fp:", lib.fp(de_vxc_off))


de_vxc fp: -16.124876249597417


### summarize of common DFT contribution


In [17]:
de_xc_recap = de_vxc_diag + de_vxc_off + de_fxc
assert np.allclose(de_xc_recap, de_ks_ref)


In [18]:
dat = dict(np.load("nh3_r_tpss0_decomp.npz"))
dat.update({
    "de_vxc_diag": de_vxc_diag,
    "de_vxc_off": de_vxc_off,
    "de_fxc": de_fxc,
})
np.savez("nh3_r_tpss0_decomp.npz", **dat)


In [19]:
de_xc_recap.sum(axis=(0, 1))


array([[-0.00403, -0.00003, -0.00001],
       [-0.00003, -0.00405,  0.     ],
       [-0.00001,  0.     , -0.00401]])

In [20]:
np.abs(de_xc_recap.sum(axis=(0, 1))).max()


np.float64(0.004050342374784605)

### becke partition derivative (preparation)


In [21]:
ngrids = grids.weights.size
ni = dft.numint.NumInt()
ao = ni.eval_ao(mol, grids.coords, deriv=3)   # deriv=3: needed for d2rho (t4/t7)
dm0 = mf.make_rdm1()
ao_dm0 = ao @ dm0
rho, exc, vxc, fxc = rks_nimatmul._eval_rho_exc_vxc_fxc("TPSS0", "MGGA", ao, ao_dm0)
drho = rks_nimatmul._make_drho("MGGA", ao, ao_dm0, mol.aoslice_by_atom())


In [22]:
natm = mol.natm
becke_scheme = grids.radii_adjust(mol, grids.atomic_radii)
adjustment_factor = np.array([becke_scheme(i, j, 0) for i in range(natm) for j in range(natm)]).reshape(natm, natm)
becke_result = becke_partition(grids.coords, mol.atom_coords(), grids.atm_idx, grids.quadrature_weights, adjustment_factor, 3, 512, 2, None)
w, dw, ddw = becke_result["w"], becke_result["dw"], becke_result["ddw"]


In [23]:
# Second-order skeleton density derivative d2rho[C, s, t, x, g] = d/dr_t of drho[C, s, x, g].
# (drho is the first skeleton derivative d rho_x / d R_{C_s}; d2rho takes one more spatial deriv.)
# Needed for the d(d rho/d r)/dB term (t4) and the double grid-shift (t7).
# pdrho = -d2rho (opposite sign vs the d2rho convention), pprho = sum_C pdrho[C] = -d2rho_sum.
IDX2 = [[XX, XY, XZ], [YX, YY, YZ], [ZX, ZY, ZZ]]
IDX3 = [
    [[XXX, XXY, XXZ], [XXY, XYY, XYZ], [XXZ, XYZ, XZZ]],
    [[XXY, XYY, XYZ], [XYY, YYY, YYZ], [XYZ, YYZ, YZZ]],
    [[XXZ, XYZ, XZZ], [XYZ, YYZ, YZZ], [XZZ, YZZ, ZZZ]],
]
pdrho = np.zeros((natm, 3, 3, 5, ngrids))
for C in range(natm):
    _, _, p0, p1 = aoslices[C]
    slc = slice(p0, p1)
    ao_slc = ao[:, :, slc]
    ao_dm0_slc = ao_dm0[:, :, slc]
    # x = 0 (rho value)
    for s in range(3):
        for t in range(3):
            term = (np.einsum("gu, gu -> g", ao_slc[IDX2[t][s]], ao_dm0_slc[O])
                  + np.einsum("gu, gu -> g", ao_slc[s + 1], ao_dm0_slc[t + 1]))
            pdrho[C, s, t, 0] += 2 * term
    # x = k+1 (sigma gradient component); needs 3rd-order AO derivatives
    for k in range(3):
        for s in range(3):
            for t in range(3):
                term = (np.einsum("gu, gu -> g", ao_slc[IDX3[t][s][k]], ao_dm0_slc[O])
                      + np.einsum("gu, gu -> g", ao_slc[IDX2[s][k]], ao_dm0_slc[t + 1])
                      + np.einsum("gu, gu -> g", ao_slc[IDX2[t][s]], ao_dm0_slc[k + 1])
                      + np.einsum("gu, gu -> g", ao_slc[s + 1], ao_dm0_slc[IDX2[t][k]]))
                pdrho[C, s, t, k + 1] += 2 * term
    # x = 4 (tau component); needs 3rd-order AO derivatives + 2nd-order ao_dm0.
    # tau does NOT carry the bra<->ket symmetry factor 2 (it is built from the
    # asymmetric (nabla bra).(nabla ket) form), so neither drho[..,4] nor d2rho[..,4] do.
    for s in range(3):
        for t in range(3):
            term = 0
            for k in range(3):
                term += (np.einsum("gu, gu -> g", ao_slc[IDX3[t][s][k]], ao_dm0_slc[k + 1])
                       + np.einsum("gu, gu -> g", ao_slc[IDX2[s][k]], ao_dm0_slc[IDX2[t][k]]))
            pdrho[C, s, t, 4] += term
pprho = pdrho.sum(axis=0)  # (s, t, x, g) = -d2 rho_x / (d r_s d r_t)


## 格点偏移二阶导数分解

回顾一阶梯度（`11-1`）中，格点偏移梯度分为两部分，且两部分大小相近、符号相反，加起来接近零（但精确描述了格点偏移增量）：

- 格点权重梯度 $T_1[A,t] = \sum_g \frac{dw_g}{dA_t} f_g$
- 泛函对格点偏移梯度 $T_2[A,t] = \sum_{g \in A} w_g\, vxc_x\, \frac{\partial \rho_x}{\partial r_{tg}}$

二阶情形完全类似。记 $dT_1 = d(T_1)/dB_s$、$dT_2 = d(T_2)/dB_s$，则格点偏移 Hessian 增量

$$\Delta H_{AB,ts} = dT_1[A,B,t,s] + dT_2[A,B,t,s]$$

两部分同样大小相近、符号相反，加起来给出小的格点偏移增量（$\sim 10^{-3}$），用以修正 `de_xc_recap` 的平动不变性（其 $\sum_{AB}$ 当前约 $4\times10^{-3}$，MGGA 比 GGA 大一个量级，主要来自 tau 通道）。

由于格点是随原子移动的，$dT_1, dT_2$ 的每一项都要做 `+= transpose` 对称化--这恰好把"骨架梯度随格点移动"的那部分贡献自动补上（除了 $A=B$ 的二阶格点偏移项 `t7`，需单独添加）。下面各项 $t_1,\dots,t_7$ 均按此约定。所有 $t_i$ 对 $x, y$ 分量的缩并自动覆盖 5 个 rho 通道（RHO, GRAD, TAU），无需对 tau 特殊处理。


### t1-t7


In [24]:
t1 = np.einsum("Atg, xg, Bsxg -> ABts", dw, vxc, drho)
t2 = np.einsum("AtBsg, g, g -> ABts", ddw, exc, rho[0])

t1 += np.einsum("ABts -> BAst", t1)


In [25]:
t3 = np.zeros((natm, natm, 3, 3))
t4 = np.zeros((natm, natm, 3, 3))
t5 = np.zeros((natm, natm, 3, 3))
t6 = np.zeros((natm, natm, 3, 3))
t7 = np.zeros((natm, natm, 3, 3))

prho = drho.sum(axis=0)
for A in range(natm):
    maskA = grids.atm_idx == A
    t3[A] -= np.einsum("g, txg, xyg, Bsyg -> Bts", weights[maskA], prho[..., maskA], fxc[..., maskA], drho[..., maskA])
    t5[A] -= np.einsum("Bsg, xg, txg -> Bts", dw[..., maskA], vxc[..., maskA], prho[..., maskA])
    t6[A, A] += np.einsum("g, xyg, syg, txg -> ts", w[maskA], fxc[..., maskA], prho[..., maskA], prho[..., maskA])
    t7[A, A] += np.einsum("g, xg, stxg -> ts", w[maskA], vxc[:, maskA], pprho[:, :, :, maskA])
    for B in range(natm):
        t4[A, B] -= np.einsum("g, xg, stxg -> ts", w[maskA], vxc[:, maskA], pdrho[B][:, :, :, maskA])

t3 += np.einsum("ABts -> BAst", t3)
t4 += np.einsum("ABts -> BAst", t4)
t5 += np.einsum("ABts -> BAst", t5)


### assemble grid-shift hessian


In [26]:
# dT1 = t1 + t2  (from the grid-weight gradient T1)
# dT2 = t3 + t4 + t5 + t6 + t7  (from the functional grid-shift gradient T2)
# The two are individually large but opposite, summing to the small grid-shift increment
# (the hessian analogue of the gradient's T1 + T2 cancellation).
dT1 = t1 + t2
dT2 = t3 + t4 + t5 + t6 + t7
grid_shift = dT1 + dT2
print("dT1 max:", np.abs(dT1).max())
print("dT2 max:", np.abs(dT2).max())
print("grid_shift max (deviation from de_xc_recap):", np.abs(grid_shift).max())
print("grid_shift sum(0,1) max:", np.abs(grid_shift.sum(axis=(0, 1))).max())


dT1 max: 0.8816034448036916
dT2 max: 0.8815051837631249
grid_shift max (deviation from de_xc_recap): 0.004226861567239504
grid_shift sum(0,1) max: 0.004050342375443106


### translational invariance: de_xc_recap + grid_shift


In [27]:
# Adding the grid-shift increment should restore translational invariance:
# sum over (A, B) of the DFT skeleton hessian drops from ~4e-3 to ~1e-13.
de_xc_grid = de_xc_recap + grid_shift
print("|de_xc_recap|.sum(0,1) max            :", np.abs(de_xc_recap.sum(axis=(0, 1))).max())
print("|de_xc_recap + grid_shift|.sum(0,1) max:", np.abs(de_xc_grid.sum(axis=(0, 1))).max())
print()
print("(np.allclose(de_xc_recap, de_xc_grid) will fail by ~1e-3, which is expected;")
print(" the grid-shift is a small correction of that magnitude.)")


|de_xc_recap|.sum(0,1) max            : 0.004050342374784605
|de_xc_recap + grid_shift|.sum(0,1) max: 3.7350261772317594e-12

(np.allclose(de_xc_recap, de_xc_grid) will fail by ~1e-3, which is expected;
 the grid-shift is a small correction of that magnitude.)


In [28]:
def _c3g3(Mdm0):
    # c3[A,B] = -2 sum_{mu in B} Mdm0[A][t,s];  g3[A,B] = -2 sum_{mu in A} Mdm0[B][s,t]
    out = np.zeros((natm, natm, 3, 3))
    for A in range(natm):
        _, _, p0A, p1A = aoslices[A]; slcA = slice(p0A, p1A)
        for B in range(natm):
            _, _, p0B, p1B = aoslices[B]; slcB = slice(p0B, p1B)
            out[A, B] += -2 * np.einsum("tsu -> ts", Mdm0[A][:, :, slcB])               # c3 (sum mu in B)
            if A == B:
                out[A, A] += 2 * np.einsum("tsu -> ts", Mdm0[A])                        # c3 A=B correction
            out[A, B] += -2 * np.einsum("tsu -> ts", Mdm0[B].transpose(1, 0, 2)[:, :, slcA])  # g3 (Mdm0[B][s,t], sum mu in A)
    return out


In [29]:
def contract_pvxc(pvxc):
    # pvxc shape: [natm, 3, 3, nao]
    de_pvxc = np.zeros((natm, natm, 3, 3))
    for A in range(natm):
        de_pvxc[A, A] += 1 * np.einsum("tsu -> ts", pvxc[A])
        for B in range(natm):
            _, _, p0B, p1B = aoslices[B]; slcB = slice(p0B, p1B)
            de_pvxc[A, B] += -2 * np.einsum("tsu -> ts", pvxc[A][:, :, slcB])
    de_pvxc += de_pvxc.transpose(1, 0, 3, 2)
    return de_pvxc


### t8


In [30]:
# --- dao_vxc_diag (MGGA: with tau) --- #  [per-atom split; mirrors cell 17]
PAIR_TS = np.array([[0, 1, 2], [1, 3, 4], [2, 4, 5]])

dao_vxc_diag = np.zeros((6, nao))  # 6 denotes xx, xy, xz, yy, yz, zz
pvxc_diag = np.zeros((natm, 3, 3, nao))

for A in range(natm):
    mA = grids.atm_idx == A

    dao_vxc_diagA = np.zeros((6, nao))

    wvA = weights[mA] * vxc[..., mA]  # [5, ngrids]
    aoA = ao[:, mA, :]
    ao_dm0A = ao_dm0[:, mA, :]

    # Contribution 1: ao[i+4]^T @ (wv[0]*ao[0] + wv[1]*ao[1] + wv[2]*ao[2] + wv[3]*ao[3])
    aow_diag = (np.einsum("gu, g -> gu", ao_dm0A[0], wvA[0])
            + np.einsum("gu, g -> gu", ao_dm0A[1], wvA[1])
            + np.einsum("gu, g -> gu", ao_dm0A[2], wvA[2])
            + np.einsum("gu, g -> gu", ao_dm0A[3], wvA[3]))
    for idx, its in enumerate([XX, XY, XZ, YY, YZ, ZZ]):
        dao_vxc_diagA[idx] += 2 * np.einsum("gu, gu -> u", aoA[its], aow_diag)

    # Contribution 2 (GGA triple-derivative part)
    TRIPLE_DERIV_DIAG = [
        [XXX, XXY, XXZ],  # xx
        [XXY, XYY, XYZ],  # xy
        [XXZ, XYZ, XZZ],  # xz
        [XYY, YYY, YYZ],  # yy
        [XYZ, YYZ, YZZ],  # yz
        [XZZ, YZZ, ZZZ],  # zz
    ]
    for idx, (i3x, i3y, i3z) in enumerate(TRIPLE_DERIV_DIAG):
        aow_triple = (np.einsum("gu, g -> gu", aoA[i3x], wvA[1])
                    + np.einsum("gu, g -> gu", aoA[i3y], wvA[2])
                    + np.einsum("gu, g -> gu", aoA[i3z], wvA[3]))
        dao_vxc_diagA[idx] += 2 * np.einsum("gu, gu -> u", aow_triple, ao_dm0A[0])

    # Contribution 3 (TAU triple-derivative part)
    aow_diag_tau = [np.einsum("gu, g -> gu", ao_dm0A[d], wvA[4]) for d in [X, Y, Z]]
    TAU_DIAG_COMPONENTS = [
        ([XXX, XXY, XXZ, XYY, XYZ, XZZ], 0),  # direction x: ao[triple]^T @ aow_tau_x
        ([XXY, XYY, XYZ, YYY, YYZ, YZZ], 1),  # direction y
        ([XXZ, XYZ, XZZ, YYZ, YZZ, ZZZ], 2),  # direction z
    ]
    for i, (triple_indices, direction) in enumerate(TAU_DIAG_COMPONENTS):
        for idx, j in enumerate(triple_indices):
            dao_vxc_diagA[idx] += np.einsum("gu, gu -> u", aoA[j], aow_diag_tau[direction])
    
    dao_vxc_diag += dao_vxc_diagA
    pvxc_diag[A] = 0.5 * dao_vxc_diagA[PAIR_TS]

de_vxc_diag = np.zeros((natm, natm, 6))
for A in range(natm):
    _, _, p0A, p1A = aoslices[A]
    slcA = slice(p0A, p1A)
    de_vxc_diag[A, A] += np.einsum("Au -> A", dao_vxc_diag[:, slcA])
de_vxc_diag = de_vxc_diag[:, :, PAIR_TS]
print("de_vxc_diag fp:", lib.fp(de_vxc_diag))

t8 = contract_pvxc(pvxc_diag)


de_vxc_diag fp: 44.683863589573626


In [31]:
# --- dao_vxc (MGGA: with tau) --- #  [per-atom split; mirrors cell 19]
wv = weights * vxc  # [5, ngrids]
dao_vxc = np.zeros((3, 3, nao, nao))
pvxc_off = np.zeros((natm, 3, 3, nao))

# GGA part (RHO + SIGMA)
GGA_CALLS = [[XX, XY, XZ], [YX, YY, YZ], [ZX, ZY, ZZ]]

for A in range(natm):
    mA = grids.atm_idx == A

    wvA = weights[mA] * vxc[..., mA]  # [5, ngrids]
    aoA = ao[:, mA, :]
    ao_dm0A = ao_dm0[:, mA, :]

    dao_vxcA = np.zeros((3, 3, nao, nao))
    aowvA = np.zeros((3, mA.sum(), nao))
    for t in range(3):
        aowvA[t] = 0.5 * np.einsum("gu, g -> gu", aoA[t + 1], wvA[0])
        for r in range(3):
            aowvA[t] += np.einsum("gu, g -> gu", aoA[GGA_CALLS[t][r]], wvA[r + 1])

    for t in range(3):
        for s in range(3):
            dao_vxcA[t, s] += 2 * aowvA[s].T @ aoA[t + 1]     # ipip[t,s]

    # TAU part (MGGA): ipip from three _d1d2_dot_ calls with wv[4]
    aowv_tauA = [np.einsum("gu, g -> gu", aoA[4 + i], wvA[4]) for i in range(6)]
    dao_vxc_tauA = np.zeros((3, 3, nao, nao))
    TAU_CALLS = [
        ([0, 1, 2], [XX, XY, XZ]),
        ([1, 3, 4], [YX, YY, YZ]),
        ([2, 4, 5], [ZX, ZY, ZZ]),
    ]
    for r_bra, r_ket in TAU_CALLS:
        for t in range(3):
            for s in range(t + 1):
                dao_vxc_tauA[t, s] += 0.5 * aowv_tauA[r_bra[s]].T @ aoA[r_ket[t]]    # ipip[d1,d2]
    for t in range(3):
        for s in range(t):
            dao_vxc_tauA[s, t] = dao_vxc_tauA[t, s].T
    dao_vxcA += dao_vxc_tauA

    dao_vxcA += dao_vxcA.transpose(1, 0, 3, 2)  # [s,t] with AO indices transposed
    
    dao_vxc += dao_vxcA
    pvxc_off[A] = 0.5 * np.einsum("tsuv, uv -> tsu", dao_vxcA, dm0)

de_vxc_off = np.zeros((natm, natm, 3, 3))
for A in range(natm):
    _, _, p0A, p1A = aoslices[A]
    slcA = slice(p0A, p1A)
    for B in range(A + 1):
        _, _, p0B, p1B = aoslices[B]
        slcB = slice(p0B, p1B)
        de_vxc_off[A, B] += np.einsum("tsuv, uv -> ts", dao_vxc[:, :, slcB, slcA], dm0[slcB, slcA])
        if A != B:
            de_vxc_off[B, A] = de_vxc_off[A, B].T
print("de_vxc fp:", lib.fp(de_vxc_off))

t9 = contract_pvxc(pvxc_off)


de_vxc fp: -16.12487624959742


In [32]:
t4t = t8 + t9
print("max|t4t - (t4 + t7)|:", np.abs(t4t - (t4 + t7)).max())


max|t4t - (t4 + t7)|: 3.950617610826157e-12


#### grid-shift hessian with d2rho fully substituted


In [33]:
# Reassemble the grid-shift hessian with t4+t7 replaced by the basis-form t4t.
# No d2rho (nor d2rho_sum) is used; only drho (in t1/t3/t5/t6) remains, as in production.
grid_shift_basis = t1 + t2 + t3 + t4t + t5 + t6
print("max|grid_shift_basis - grid_shift|:", np.abs(grid_shift_basis - grid_shift).max())
print("|de_ks_ref + grid_shift_basis|.sum(0,1) max:",
      np.abs((de_ks_ref + grid_shift_basis).sum(axis=(0, 1))).max())


max|grid_shift_basis - grid_shift|: 3.944955473400569e-12
|de_ks_ref + grid_shift_basis|.sum(0,1) max: 1.6775469902086115e-12


In [34]:
# recheck if the non-grid-shift part of the hessian is still correct (it should be)
de_xc_recap = de_vxc_diag + de_vxc_off + de_fxc
assert np.allclose(de_xc_recap, de_ks_ref)
